<a href="https://colab.research.google.com/github/VickyW2366/Shors-optimisations/blob/main/Optimization_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Noise-Resilient Post-Processing

"""
 Instead of simple continued fractions, use multiple samples and robust estimation.

Current Post-Processing:
 Simple continued fractions - single measurement
measured_phases = []
for output, count in counts.items():
    phase = int(output, 2) / (2**n_count)
    frac = Fraction(phase).limit_denominator(N)
    r = frac.denominator
    possible_r.append((r, count))
 """
# Optimized with Robust Estimation:

#!pip install qiskit
#!pip install qiskit-aer
import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.circuit.library import QFT
from qiskit.visualization import plot_histogram
from math import gcd
from fractions import Fraction
from sympy.ntheory import factorint
import random
from scipy.stats import norm
import numpy as np
import time

# Measures how long the program takes to run
start_time = time.monotonic()

def robust_period_estimation(counts, n_count, N, confidence=0.95):
    """
    Robust period estimation using multiple measurements and statistical methods.
    Optimizations:
    1. Weighted histogram of measured phases
    2. Gaussian kernel density estimation for peak finding
    3. Statistical validation of candidate periods
    """
    # Extract all measured phases with their frequencies
    phases = []
    weights = []

    for output, count in counts.items():
        phase = int(output, 2) / (2**n_count)
        if phase > 0:  # Exclude zero phase
            phases.append(phase)
            weights.append(count)

    # Convert to numpy arrays
    phases = np.array(phases)
    weights = np.array(weights)
    weights = weights / weights.sum()  # Normalize

    # Use kernel density estimation to find peaks
    from sklearn.neighbors import KernelDensity
    kde = KernelDensity(kernel='gaussian', bandwidth=0.05)
    kde.fit(phases.reshape(-1, 1), sample_weight=weights)

    # Find peaks in the density estimate
    x_plot = np.linspace(0, 1, 1000).reshape(-1, 1)
    log_dens = kde.score_samples(x_plot)
    density = np.exp(log_dens)

    # Find local maxima
    from scipy.signal import find_peaks
    peaks, properties = find_peaks(density, height=0.1*np.max(density))
    peak_phases = x_plot[peaks].flatten()

    # For each peak, find candidate periods
    candidates = []
    for phase in peak_phases:
        # Try all possible denominators up to N
        for denominator in range(1, N+1):
            approx_phase = round(phase * denominator) / denominator
            if abs(approx_phase - phase) < 0.01:  # Close enough
                candidates.append(denominator)
    # Count frequency of each candidate period
    from collections import Counter
    period_counts = Counter(candidates)

    # Return most common period
    if period_counts:
        most_common = period_counts.most_common(1)[0][0]

        # Validation step
        if most_common > 1 and most_common < N:
            # Check if period divides N-1 (from group theory)
            if (N-1) % most_common == 0:
                print(f"✓ Validated period: {most_common}")
                return most_common
            else:
                print(f"x Period {most_common} not validated, continuing")

    return None

# Simulate a quantum circuit to get measurement counts
# Example: A simple circuit that measures 3 qubits
n_qubits = 3
qc_example = QuantumCircuit(n_qubits, n_qubits) # n_qubits quantum bits, n_qubits classical bits
qc_example.h(0) # Apply a Hadamard gate to qubit 0 for superposition
qc_example.cx(0, 1) # CNOT gate
qc_example.cx(0, 2) # CNOT gate
qc_example.measure(range(n_qubits), range(n_qubits)) # Measure all qubits

# Simulates the circuit
simulator = Aer.get_backend('qasm_simulator')
transpiled_qc = transpile(qc_example, simulator)
job = simulator.run(transpiled_qc, shots=1024)
result = job.result()
counts = result.get_counts(qc_example)

# Calls the function
estimated_period = robust_period_estimation(counts, n_count=3, N=5)
print(f"Estimated period: {estimated_period}")
print(qc_example.draw())
print("Program took %s seconds to run" % (time.time() - start_time))

Estimated period: None
     ┌───┐             ┌─┐   
q_0: ┤ H ├──■────■─────┤M├───
     └───┘┌─┴─┐  │  ┌─┐└╥┘   
q_1: ─────┤ X ├──┼──┤M├─╫────
          └───┘┌─┴─┐└╥┘ ║ ┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     1  0  2 
Program took 1777847156.9939773 seconds to run
